# Benchmarks - JavaScript

The one JavaScript example from [docs/benchmarks.md](https://platob.github.io/yggdryl/benchmarks/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

### The pipeline those numbers measure

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const zlib = require('node:zlib')
const { IOBase, iceberg, schemaFromPattern, zstd } = require('yggdryl')

const pattern =
  '^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}\\S*' +
  ' \\[(?<level>[^\\]]+)\\] \\[(?<logger>[^\\]]+)\\]' +
  ' \\[(?<thread_id>\\d+)\\] took=(?<latency_us>\\d+)'
// The older extractor: same records, no thread column.
const archivedPattern = pattern.replace('\\[(?<thread_id>\\d+)\\]', '\\[\\d+\\]')

const extractor = {
  pattern,
  byteSize: 8 * 1024 * 1024,
  customFields: { source: 'gateway' },
}
const older = { pattern: archivedPattern, customFields: { source: 'archive' } }

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-'))
const incoming = path.join(root, 'incoming')
const archive = path.join(root, 'archive')
fs.mkdirSync(incoming)
fs.mkdirSync(archive)

// Three rotated leaves in three codings; the second record spans a stack trace.
fs.writeFileSync(
  path.join(incoming, 'app-0.log.gz'),
  zlib.gzipSync(
    Buffer.from(
      '2024-02-01 10:00:00.000000 [ii] [engine] [3] took=120 fill 100 SYMB-0001\n' +
        '2024-02-01 10:00:01.000000 [ee] [engine] [4] took=980 fill 101 SYMB-0002\n' +
        '    at engine::match(order.rs:118)\n' +
        '    at engine::step(order.rs:64)\n' +
        '2024-02-01 10:00:02.000000 [ww] [router] [5] took=240 fill 102 SYMB-0003\n',
    ),
  ),
)
fs.writeFileSync(
  path.join(incoming, 'app-1.log'),
  '2024-02-01 10:00:03.000000 [ee] [ledger] [6] took=770 fill 103 SYMB-0004\n',
)
fs.writeFileSync(
  path.join(incoming, 'app-2.log.zst'),
  zstd.dumps(
    Buffer.from('2024-02-01 10:00:04.000000 [ii] [feed] [7] took=100 fill 104 SYMB-0005\n'),
  ),
)
fs.writeFileSync(
  path.join(archive, 'app-9.log.gz'),
  zlib.gzipSync(
    Buffer.from('2024-01-31 23:59:59.000000 [ee] [engine] [2] took=310 fill 099 SYMB-0000\n'),
  ),
)

// 1. The table exists before the first record does.
const marked = schemaFromPattern(extractor).withPartitionFields(['level'])
const catalog = new iceberg.Catalog(path.join(root, 'warehouse'))
const table = catalog.tables.create('logs.app', marked)

// 2, 3, 4. One handle per folder, and one lazy combine over the two.
const stream = new IOBase(incoming)
  .readArrowLines(extractor)
  .combined(new IOBase(archive).readArrowLines(older))

// 5. One commit, handed the reader itself - never an array of batches.
table.append(stream)

// The read-back asserts on the table, not on anything held in memory.
const rows = table.scan().toTable()
// Five live records - not the seven lines they occupy - and one archived.
assert.equal(rows.numRows, 6)
const latency = [...rows.getChild('latency_us')].reduce((total, took) => total + took, 0n)
assert.equal(latency, 120n + 980n + 240n + 770n + 100n + 310n)
assert.equal(rows.getChild('thread_id').nullCount, 1)
assert.deepEqual(
  new Set(rows.getChild('source')),
  new Set(['gateway', 'archive']),
)

fs.rmSync(root, { recursive: true, force: true })